# Analyse de la Pollution Atmospherique - UrbanHub

> **Role 3 - Qualite de l'Air** · Source : OpenAQ · 6 villes francaises

Ce notebook interactif presente une analyse visuelle animee de la qualite de l'air avec **Plotly**.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = "plotly_dark"

CITY_COLORS = {
    "paris":     "#EF476F",
    "lyon":      "#FFD166",
    "marseille": "#06D6A0",
    "toulouse":  "#118AB2",
    "bordeaux":  "#A855F7",
    "lille":     "#F97316",
}

WHO_THRESHOLDS = {
    "avg_pm25": 15,
    "avg_pm10": 45,
    "avg_no2":  25,
    "avg_o3":   100,
    "avg_co":   4000,
}

def hex_to_rgba(hex_color, alpha=0.15):
    """Convertit #RRGGBB en rgba(r,g,b,alpha)"""
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

print("Librairies chargees - OK")

## 1. Chargement des donnees Gold

In [ ]:
gold_dir = os.path.join("..", "data", "urbanhub-gold", "pollution")
files = glob.glob(os.path.join(gold_dir, "city=*", "daily_aggregates.parquet"))

if not files:
    raise FileNotFoundError("Aucun fichier Gold trouve. Executez d'abord le pipeline !")

dfs = [pd.read_parquet(f) for f in files]
df = pd.concat(dfs, ignore_index=True)

df["city"] = df["city"].str.lower().str.strip()
df["date"] = pd.to_datetime(df["date"])
df["city_label"] = df["city"].str.capitalize()

for col, seuil in WHO_THRESHOLDS.items():
    if col in df.columns:
        df[f"{col}_ratio"] = (df[col] / seuil * 100).round(1)

aqi_cols = [c for c in df.columns if c.endswith("_ratio")]
df["aqi_score"] = df[aqi_cols].max(axis=1).round(1)

print(f"{len(df)} enregistrements - {df['city'].nunique()} villes")
df[["city_label", "avg_pm25", "avg_pm10", "avg_no2", "avg_o3", "avg_co", "pollution_episode_flag", "aqi_score"]]

## 2. Jauges AQI par ville

In [ ]:
df_sorted = df.sort_values("aqi_score", ascending=False).reset_index(drop=True)
cities = df_sorted["city"].tolist()

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[c.capitalize() for c in cities],
    specs=[[{"type": "indicator"}]*3]*2
)

positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]

for i, city in enumerate(cities):
    row_data = df_sorted[df_sorted["city"] == city].iloc[0]
    score = row_data["aqi_score"]
    color = CITY_COLORS.get(city, "#888888")
    r, c = positions[i]

    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=score,
        number={"suffix": "%", "font": {"size": 26, "color": color}},
        gauge={
            "axis": {"range": [0, 200], "tickwidth": 1},
            "bar": {"color": color, "thickness": 0.3},
            "bgcolor": "rgba(255,255,255,0.05)",
            "borderwidth": 0,
            "steps": [
                {"range": [0, 50],    "color": "rgba(6,214,160,0.15)"},
                {"range": [50, 100],  "color": "rgba(255,209,102,0.15)"},
                {"range": [100, 200], "color": "rgba(239,71,111,0.15)"},
            ],
            "threshold": {
                "line": {"color": "white", "width": 3},
                "thickness": 0.75,
                "value": 100
            }
        },
        title={"text": "vs seuil OMS", "font": {"size": 11}}
    ), row=r, col=c)

fig.update_layout(
    title=dict(text="Indice AQI simplifie - % du seuil OMS maximal", font=dict(size=20, color="white")),
    height=480,
    paper_bgcolor="rgba(15,15,25,1)",
    font=dict(color="white")
)
fig.show()

## 3. Comparaison des polluants par ville

In [ ]:
polluants = ["avg_pm25", "avg_pm10", "avg_no2", "avg_o3"]
labels    = ["PM2.5", "PM10", "NO2", "O3"]
seuils    = [WHO_THRESHOLDS[p] for p in polluants]

df_long = df.melt(
    id_vars=["city_label"],
    value_vars=polluants,
    var_name="polluant",
    value_name="valeur"
)
df_long["label"] = df_long["polluant"].map(dict(zip(polluants, labels)))
df_long["seuil"] = df_long["polluant"].map(dict(zip(polluants, seuils)))

color_map = {c.capitalize(): CITY_COLORS[c] for c in CITY_COLORS}

fig = px.bar(
    df_long.sort_values(["label", "valeur"], ascending=[True, False]),
    x="city_label",
    y="valeur",
    color="city_label",
    color_discrete_map=color_map,
    facet_col="label",
    facet_col_wrap=2,
    labels={"valeur": "Concentration (ug/m3)", "city_label": "Ville"},
    title="Niveaux de polluants par ville",
    template="plotly_dark",
    height=700
)

# Lignes de seuil OMS
for i, (seuil, label) in enumerate(zip(seuils, labels)):
    row = (i // 2) + 1
    col = (i % 2) + 1
    fig.add_hline(
        y=seuil,
        line_dash="dash",
        line_color="rgba(239,71,111,0.8)",
        line_width=2,
        annotation_text=f"OMS: {seuil}",
        annotation_font_color="#EF476F",
        row=row, col=col
    )

fig.update_layout(
    paper_bgcolor="rgba(15,15,25,1)",
    plot_bgcolor="rgba(25,25,40,1)",
    showlegend=False,
    font=dict(color="white")
)
fig.show()

## 4. Radar Chart - Profil polluant par ville

In [ ]:
radar_polluants = ["avg_pm25", "avg_pm10", "avg_no2", "avg_o3", "avg_co"]
radar_labels    = ["PM2.5", "PM10", "NO2", "O3", "CO"]

fig = go.Figure()

for _, row_data in df.iterrows():
    city  = row_data["city"]
    color = CITY_COLORS.get(city, "#888888")
    values = [(row_data[p] / WHO_THRESHOLDS[p] * 100) for p in radar_polluants]
    # Fermer le polygone
    values_closed = values + [values[0]]
    labels_closed = radar_labels + [radar_labels[0]]

    fig.add_trace(go.Scatterpolar(
        r=values_closed,
        theta=labels_closed,
        fill="toself",
        fillcolor=hex_to_rgba(color, 0.15),
        line=dict(color=color, width=2.5),
        name=city.capitalize(),
        hovertemplate="<b>%{theta}</b><br>%{r:.1f}% du seuil OMS<extra></extra>"
    ))

# Cercle de reference OMS a 100%
theta_ref = radar_labels + [radar_labels[0]]
fig.add_trace(go.Scatterpolar(
    r=[100] * len(theta_ref),
    theta=theta_ref,
    line=dict(color="rgba(239,71,111,0.7)", width=2, dash="dash"),
    mode="lines",
    name="Seuil OMS",
    showlegend=True
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 200],
            gridcolor="rgba(255,255,255,0.1)",
            tickfont=dict(size=9, color="rgba(255,255,255,0.5)"),
            ticksuffix="%"
        ),
        angularaxis=dict(
            gridcolor="rgba(255,255,255,0.1)",
            tickfont=dict(size=13, color="white")
        ),
        bgcolor="rgba(25,25,40,1)"
    ),
    title=dict(text="Radar - Profil polluant par ville (% seuil OMS)", font=dict(size=20, color="white")),
    paper_bgcolor="rgba(15,15,25,1)",
    font=dict(color="white"),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.08),
    height=580
)
fig.show()

## 5. Carte a bulles - Vue geographique

In [ ]:
CITY_COORDS = {
    "paris":     {"lat": 48.853, "lon": 2.349},
    "lyon":      {"lat": 45.748, "lon": 4.847},
    "marseille": {"lat": 43.296, "lon": 5.381},
    "toulouse":  {"lat": 43.605, "lon": 1.444},
    "bordeaux":  {"lat": 44.837, "lon": -0.580},
    "lille":     {"lat": 50.629, "lon": 3.057},
}

df["lat"] = df["city"].map(lambda c: CITY_COORDS.get(c, {}).get("lat"))
df["lon"] = df["city"].map(lambda c: CITY_COORDS.get(c, {}).get("lon"))

polluants_map = ["avg_pm25", "avg_pm10", "avg_no2", "avg_o3"]
labels_map    = ["PM2.5", "PM10", "NO2", "O3"]

frames_list = []
for p, lbl in zip(polluants_map, labels_map):
    seuil = WHO_THRESHOLDS[p]
    tmp = df[["city", "city_label", "lat", "lon", p]].copy()
    tmp.rename(columns={p: "valeur"}, inplace=True)
    tmp["polluant"] = lbl
    tmp["seuil"]   = seuil
    tmp["taille"]  = (tmp["valeur"] / seuil * 40 + 8).clip(8, 80)
    frames_list.append(tmp)

df_map = pd.concat(frames_list, ignore_index=True)
color_map_cap = {c.capitalize(): CITY_COLORS[c] for c in CITY_COLORS}

fig = px.scatter_mapbox(
    df_map,
    lat="lat", lon="lon",
    size="taille",
    color="city_label",
    color_discrete_map=color_map_cap,
    hover_name="city_label",
    hover_data={"lat": False, "lon": False, "taille": False, "valeur": True, "seuil": True},
    animation_frame="polluant",
    mapbox_style="carto-darkmatter",
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    title="Carte animee - Pollution par polluant",
    height=580
)

fig.update_layout(
    paper_bgcolor="rgba(15,15,25,1)",
    font=dict(color="white", size=13)
)
fig.show()

## 6. Heatmap - Intensite relative des polluants

In [ ]:
hp = ["avg_pm25", "avg_pm10", "avg_no2", "avg_o3", "avg_co"]
hl = ["PM2.5", "PM10", "NO2", "O3", "CO"]

df_heat = df.set_index("city_label")[hp].copy()
df_heat.columns = hl

# Z-score par colonne pour normaliser
df_norm = (df_heat - df_heat.mean()) / (df_heat.std() + 1e-9)

fig = go.Figure(data=go.Heatmap(
    z=df_norm.values,
    x=hl,
    y=df_heat.index.tolist(),
    colorscale=[
        [0.0, "#0a0a1a"],
        [0.3, "#118AB2"],
        [0.6, "#FFD166"],
        [0.85, "#EF476F"],
        [1.0, "#ff1a4a"]
    ],
    colorbar=dict(title="Score", titlefont=dict(color="white"), tickfont=dict(color="white")),
    text=df_heat.values.round(1),
    texttemplate="%{text}",
    textfont=dict(size=13, color="white"),
    hovertemplate="<b>%{y}</b> - %{x}<br>Valeur: %{text}<br>Score normalise: %{z:.2f}<extra></extra>"
))

fig.update_layout(
    title=dict(text="Heatmap - Intensite relative des polluants par ville", font=dict(size=18, color="white")),
    paper_bgcolor="rgba(15,15,25,1)",
    plot_bgcolor="rgba(15,15,25,1)",
    font=dict(color="white"),
    xaxis=dict(tickfont=dict(size=14, color="white")),
    yaxis=dict(tickfont=dict(size=13, color="white")),
    height=400
)
fig.show()

## 7. Classement AQI et episodes de pollution

In [ ]:
df_rank = df.sort_values("aqi_score", ascending=True).reset_index(drop=True)

fig = go.Figure()

for _, row_data in df_rank.iterrows():
    city = row_data["city"]
    color = CITY_COLORS.get(city, "#888888")
    is_ep = bool(row_data["pollution_episode_flag"])
    label_ep = "EPISODE" if is_ep else "OK"

    fig.add_trace(go.Bar(
        x=[row_data["aqi_score"]],
        y=[row_data["city_label"]],
        orientation="h",
        marker=dict(
            color=color,
            line=dict(
                color="#EF476F" if is_ep else "rgba(255,255,255,0.2)",
                width=3 if is_ep else 1
            )
        ),
        text=f"{row_data['aqi_score']:.1f}%  [{label_ep}]",
        textposition="outside",
        textfont=dict(size=12, color="white"),
        hovertemplate=(
            f"<b>{row_data['city_label']}</b><br>"
            f"AQI: {row_data['aqi_score']:.1f}%<br>"
            f"PM2.5: {row_data['avg_pm25']:.2f}<br>"
            f"PM10: {row_data['avg_pm10']:.2f}<br>"
            f"NO2: {row_data['avg_no2']:.2f}<br>"
            f"Episode: {label_ep}"
            "<extra></extra>"
        ),
        showlegend=False
    ))

fig.add_vline(
    x=100,
    line_dash="dash",
    line_color="rgba(239,71,111,0.8)",
    line_width=2,
    annotation_text="Seuil OMS",
    annotation_font_color="#EF476F",
    annotation_position="top"
)

fig.update_layout(
    title=dict(text="Classement AQI par ville - Episodes de pollution", font=dict(size=18, color="white")),
    xaxis=dict(title="Indice AQI (%)", range=[0, 260], gridcolor="rgba(255,255,255,0.08)", tickfont=dict(color="white")),
    yaxis=dict(tickfont=dict(size=14, color="white")),
    paper_bgcolor="rgba(15,15,25,1)",
    plot_bgcolor="rgba(25,25,40,1)",
    font=dict(color="white"),
    height=400,
    bargap=0.35
)
fig.show()

## 8. Tableau de synthese

In [ ]:
cols = ["city_label", "avg_pm25", "avg_pm10", "avg_no2", "avg_o3", "avg_co", "aqi_score", "pollution_episode_flag"]
df_display = df[cols].sort_values("aqi_score", ascending=False).copy()
df_display.columns = ["Ville", "PM2.5", "PM10", "NO2", "O3", "CO", "AQI (%)", "Episode"]
df_display["Episode"] = df_display["Episode"].map({True: "OUI", False: "NON"})

ville_colors = [hex_to_rgba(CITY_COLORS.get(v.lower(), "#888888"), 0.15) for v in df_display["Ville"]]
cell_colors  = [ville_colors] + [["rgba(20,20,40,0.8)"] * len(df_display)] * (len(df_display.columns) - 1)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f"<b>{c}</b>" for c in df_display.columns],
        fill_color="rgba(20,20,40,1)",
        font=dict(color="white", size=14),
        line_color="rgba(255,255,255,0.1)",
        align="center",
        height=40
    ),
    cells=dict(
        values=[df_display[col].tolist() for col in df_display.columns],
        fill_color=cell_colors,
        font=dict(color="white", size=13),
        line_color="rgba(255,255,255,0.07)",
        align=["left"] + ["center"] * (len(df_display.columns) - 1),
        height=36,
        format=[None, ".2f", ".2f", ".2f", ".2f", ".1f", ".1f", None]
    )
)])

fig.update_layout(
    title=dict(text="Synthese - Qualite de l'air par ville", font=dict(size=18, color="white")),
    paper_bgcolor="rgba(15,15,25,1)",
    font=dict(color="white"),
    height=330
)
fig.show()

print("Analyse terminee !")